In [ ]:
#URL : https://www.kaggle.com/competitions/playground-series-s6e3/overview

In [2]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder,OrdinalEncoder,LabelEncoder,StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer
from sklearn.model_selection import cross_val_score, KFold
from xgboost import XGBClassifier
import warnings

In [3]:
warnings.simplefilter(action='ignore', category=FutureWarning)

In [4]:
X_train = pd.read_csv('https://huggingface.co/datasets/NHANGIOI/Learning-Training/resolve/main/playground-series-s6e3/train.csv',index_col = 'id')
X_test = pd.read_csv('https://huggingface.co/datasets/NHANGIOI/Learning-Training/resolve/main/playground-series-s6e3/test.csv',index_col = 'id')
y_train = X_train['Churn']
X_train.drop(columns = 'Churn',inplace = True)

In [5]:
X_train.columns

Index(['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
       'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
       'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
       'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod',
       'MonthlyCharges', 'TotalCharges'],
      dtype='object')

In [ ]:
print([col for col in X_train.columns if X_train[col].dtype in ['int','float']])

['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']


<function print(*args, sep=' ', end='\n', file=None, flush=False)>

In [7]:
num_cols = ['tenure','MonthlyCharges','TotalCharges']
cat_Or_cols = ['MultipleLines','OnlineSecurity','OnlineBackup','DeviceProtection','TechSupport','StreamingTV','StreamingMovies']
cat_Bin_cols = ['gender','SeniorCitizen','Partner','Dependents','PhoneService','PaperlessBilling']
cat_OH_cols = ['InternetService','Contract','PaymentMethod',]

In [ ]:
def binary_map(x):
    return x.replace({
        'Yes' : 1,
        'No' : 0,
        'Male' : 1,
        'Female' : 0,
        1 : 1,
        0 : 0
    })
Binary_Encoder = FunctionTransformer(binary_map,check_inverse = False,validate = False)

preprocessing = ColumnTransformer(transformers = [
    ('num',StandardScaler(copy = True,with_mean = True,with_std = True),num_cols),
    ('cat_binary',Binary_Encoder,cat_Bin_cols),
    ('cat_Or',OrdinalEncoder(dtype = int,handle_unknown = 'use_encoded_value',unknown_value = -1),cat_Or_cols),
    ('cat_Oh',OneHotEncoder(categories = 'auto',sparse_output = True,handle_unknown = 'ignore'),cat_OH_cols)
])
y_train = y_train.map({'Yes' : 1,'No' : 0})

In [14]:
cnt_0 = 0
for i in range(y_train.shape[0]):
    if y_train.loc[i] == 0:
        cnt_0 += 1
print(cnt_0)

460377


In [17]:
model = Pipeline(steps = [
    ('preprocess',preprocessing),
    ('model',XGBClassifier(
        n_estimators = 1500,
        max_depth=8,
        learning_rate = 0.036,
        scale_pos_weight = cnt_0 / (y_train.shape[0] - cnt_0),
        reg_alpha = 0,
        reg_lambda = 1,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
    ))
])

In [10]:
kf = KFold(n_splits = 10, shuffle = True, random_state = 42)
scores = cross_val_score(
    model,
    X_train,
    y_train,
    cv = kf,
    scoring = 'neg_log_loss'
)
print(-scores.mean())

0.3001533781791326


In [18]:
model.fit(X_train,y_train)

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['tenure', 'MonthlyCharges',
                                                   'TotalCharges']),
                                                 ('cat_binary',
                                                  FunctionTransformer(func=<function binary_map at 0x7fdf46e4ccc0>),
                                                  ['gender', 'SeniorCitizen',
                                                   'Partner', 'Dependents',
                                                   'PhoneService',
                                                   'PaperlessBilling']),
                                                 ('cat_Or',
                                                  OrdinalEncoder(dtype=<class 'int'>,
                                                                 handle_unknown...
                               feature_types=None, feature_weights=None,
                               gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None,
                               learning_rate=0.036, max_bin=None,
                               max_cat_threshold=None, max_cat_to_onehot=None,
                               max_delta_step=None, max_depth=8,
                               max_leaves=None, min_child_weight=None,
                               missing=nan, monotone_constraints=None,
                               multi_strategy=None, n_estimators=1500,
                               n_jobs=-1, num_parallel_tree=None, ...))])

In [19]:
preds = model.predict(X_test)
submission = pd.DataFrame({
    'id' : X_test.index,
    'Churn' : preds
})
submission.to_csv('submission.csv',index = False)